# Старая, но новая структура

В ноутбуке [RAGv2](https://colab.research.google.com/drive/1i6jDMmelzyh1Gxe_RGxTOL4tqA0GN_ei?usp=sharing) описана структура, которая предполагалась для локального развертывания. Технически она полностью работает, однако локально развернуть ее становится проблемой из-за требуемых мощностей для модели. В нашем случае ОЗУ 16ГБ было недостаточно - процесс предварительной индексации убивался хостом, а развертывание сервиса с моделью грузило ресурсы компьютера до предела и вызывало критическое отключение процесса выполнения из-за переполнения ОЗУ.

Поэтому было принято решение вернуться к реализации схемы с отправкой запросов по API к моделям на серверах OpenRouter. В таком случае у нас становится значительно больше возможностей для работы с моделями, а собственные ресурсы при этом становятся намного более назначимыми.
С точки зрения бизнес-логики, это плюс, поскольку такая структура сервиса позволяет запускать его практически на любом железе.


### Библиотеки

In [ ]:
!pip install -q sentence-transformers faiss-cpu rank_bm25 transformers torch pandas numpy requests python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 74.6 MB/s eta 0:00:00


## Загрузка документов

Предварительно документы были предобработаны при помощи ноутбука [EDA](https://colab.research.google.com/drive/1FJ5ecsCTVu42F6XH0Me_fPpNnf0bUoce?usp=sharing), где они были приведены к общему формату, то есть все файлы были преобразованы в .txt. Помимо этого были убраны артефакты файлов, такие как лишние табуляции и пробелы, разорванные слова и прочее.

Очищенные документы подаются на вход для дальнейшей обработки и представления модели.

In [ ]:
import json
from pathlib import Path

CLEAN_DOCS_DIR = Path("clean_docs")  # путь к папке с очищенными текстами

documents = []
for txt_path in CLEAN_DOCS_DIR.glob("*.txt"):
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read()
    documents.append({"id": txt_path.stem, "text": text})

print(f"Загружено {len(documents)} документов")
if documents:
    print("Пример (первые 200 символов):")
    print(documents[0]["text"][:200])

Загружено 15 документов
Пример (первые 200 символов):
V Veracier UK Ltd Innovation Park, Filton Bristol BS34 7QN, United Kingdom Companies House No. 08932156 TVA: GB 312 456 789 Veracier UK Ltd - Companies House No. Tel: +44 117 903 4500 | veracier.co.uk


## Чанки

Разбиваем текст на чанки размером 512 с перекрытием 64. Разделение на чанки будет по возможности по границе предложения, то есть после точки (это делается для того, чтобы было меньше проблем с потерей контекста в чанках).

In [ ]:
from typing import List

def recursive_split(text, chunk_size=512, chunk_overlap=64, separators=":;"):
    if separators == ":;":
        separators = ["\n\n", "\n", " ", ""]
    chunks = []
    start_index = 0
    while start_index < len(text):
        potential_chunk_end = min(start_index + chunk_size, len(text))
        split_at = -1
        for sep in separators:
            temp_split_at = text.rfind(sep, start_index, potential_chunk_end)
            if temp_split_at != -1:
                split_at = temp_split_at
                break
        if split_at == -1:
            split_at = potential_chunk_end
        if split_at == start_index and start_index < len(text):
            split_at = min(start_index + 1, len(text))
        current_chunk = text[start_index:split_at].strip()
        if current_chunk:
            chunks.append(current_chunk)
        next_start_index = max(start_index + 1, split_at - chunk_overlap)
        start_index = next_start_index
    return chunks

# Применяем ко всем документам
all_chunks = []
for doc in documents:
    split_chunks = recursive_split(doc["text"])
    for i, ch in enumerate(split_chunks):
        all_chunks.append({"id": f"{doc['id']}-{i}", "text": ch})

print(f"Создано {len(all_chunks)} чанков")
print("Пример чанка (первые 300 символов):")
if all_chunks:
    print(all_chunks[0]["text"][:300])

Создано 1529 чанков
Пример чанка (первые 300 символов):
V Veracier UK Ltd Innovation Park, Filton Bristol BS34 7QN, United Kingdom Companies House No. 08932156 TVA: GB 312 456 789 Veracier UK Ltd - Companies House No. Tel: +44 117 903 4500 | veracier.co.uk PLAN DE CHARGE Plan de Charge The Supplier shall maintain a traceability system enabling identifica


### Создание FAISS-индекса (dense поиск)

Для эмбеддингов используем модель all-MiniLM-L6-v2. Она хорошо справляется с эмбеддингами, проверена опытом, а именно была использована для аналогичной задачи в [прошлом проекте](https://github.com/Alisa-gh2/first_llm_wth_rag).

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Загрузка модели эмбеддингов...")
dense_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_texts = [c["text"] for c in all_chunks]

print("Кодирование чанков...")
dense_embeddings = dense_model.encode(chunk_texts, show_progress_bar=True)
dim = dense_embeddings.shape[1]

# индекс для косинусного сходства
index = faiss.IndexFlatIP(dim)
faiss.normalize_L2(dense_embeddings)
index.add(dense_embeddings)

print(f"FAISS индекс создан, размерность {dim}, добавлено {index.ntotal} векторов")

Загрузка модели эмбеддингов...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Кодирование чанков...


Batches:   0%|          | 0/48 [00:00<?, ?it/s]

FAISS индекс создан, размерность 384, добавлено 1529 векторов


### Создадим BM25-индекс (sparse поиск)

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [text.split() for text in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)
print("BM25 индекс создан")

BM25 индекс создан


## Гибридный поиск

Мы бы могли использовать здесь взвешенную сумму (alpha = 0.6), но возьмем RRF (Reciprocal Rank Fusion).

RRF лучше alpha тем. что не требует нормальизации оценок, а также работате исключительно с рангами в отличие от alpha. Он более универсален и дает более сбалансированный результат вне зависимости от того, что мы используем: dense или sparse - результат будет одинаково корректным и сбалансированным. Нам требуется только задать константку k, и то она не сильно влияет на качество поиска.


In [ ]:
def hybrid_search_rrf(query, top_k=15, k_rrf=60):
    # Dense поиск (FAISS)
    q_emb = dense_model.encode([query])
    faiss.normalize_L2(q_emb)
    dense_scores, dense_indices = index.search(q_emb, top_k)
    dense_ranks = {}
    for rank, idx in enumerate(dense_indices[0]):
        if idx != -1 and idx < len(all_chunks):
            dense_ranks[int(idx)] = rank + 1

    # BM25 поиск
    bm25_scores = bm25.get_scores(query.split())
    bm25_top_indices = np.argsort(bm25_scores)[-top_k:][::-1]
    bm25_ranks = {int(idx): rank+1 for rank, idx in enumerate(bm25_top_indices) if 0 <= idx < len(all_chunks)}

    # RRF слияние
    rrf_scores = {}
    for idx, rank in dense_ranks.items():
        rrf_scores[idx] = 1 / (k_rrf + rank)
    for idx, rank in bm25_ranks.items():
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k_rrf + rank)

    # Сортируем по убыванию RRF скора
    sorted_indices = sorted(rrf_scores.keys(), key=lambda i: rrf_scores[i], reverse=True)
    return [(all_chunks[idx], rrf_scores[idx]) for idx in sorted_indices[:top_k]]

## Переранжирование

Чтобы повысить точность релевантности используем кросс-енкодер cross-endocer/ms-macro-MiniLM-L-6-v2. Он работает так, что обрабатывает пару значений запрос-чанк, и выдает рейтинг релевантности, по нему отбирается позже итоговый топ-5.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

print("Загрузка модели переранжирования...")
rerank_model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
rerank_tokenizer = AutoTokenizer.from_pretrained(rerank_model_name)
rerank_model = AutoModelForSequenceClassification.from_pretrained(rerank_model_name)
rerank_model.eval()

def rerank(query, chunks_with_scores, top_n=5):
    if not chunks_with_scores:
        return []
    pairs = [(query, chunk["text"]) for chunk, _ in chunks_with_scores]
    inputs = rerank_tokenizer(pairs, padding=True, truncation=True, return_tensors="pt", max_length=512)
    with torch.no_grad():
        scores = rerank_model(**inputs).logits.squeeze().tolist()
    if isinstance(scores, float):
        scores = [scores]
    reranked = sorted(zip(chunks_with_scores, scores), key=lambda x: x[1], reverse=True)
    return [chunk for (chunk, _), _ in reranked[:top_n]]

Загрузка модели переранжирования...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Генерация ответа

## Загрузка переменных окружения и метод для генерации ответа

In [ ]:
!pip install OpenRouter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 13.2 MB/s eta 0:00:00


Добавляем API-ключ от своего OpenRouter. По нему мы будем обращаться к модели.

Для нашей задачи была выбрана модель z-ai/glm-4.5-air:free как сравнительно быстрая при сохранении качества. Она хорошо подходит для RAG-сервисов, подобных нашему.

Также были протестированы google-gemma-4:free и meta-llama/llama-3.2-3b-instruct:free, однако было определено, что по балансу "время ответа - качество ответа" лидерство одержала z-ai/glm-4.5-air:free.

Промпт для модели был составлен такой:
```
Ты — ассистент, отвечающий строго на основе предоставленного контекста.
Контекст состоит из фрагментов документов. Твои шаги:
1. Прочитай внимательно каждый фрагмент.
2. Найди конкретные факты, связанные с вопросом.
3. Составь краткий ответ, используя только эти факты (можно цитировать).
4. Если в контексте недостаточно информации для ответа, напиши ровно: 'Информации в предоставленных документах недостаточно.'
Не добавляй ничего от себя.
```

Ответ генерируется моделью на сервере OpenRouter.

In [ ]:
from openrouter import OpenRouter
from getpass import getpass

OPENROUTER_API_KEY = OPENROUTER_API_KEY = getpass("Введите ваш OpenRouter API key: ")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "z-ai/glm-4.5-air:free"

# Запрашиваем API‑ключ, если ещё не задан
if 'OPENROUTER_API_KEY' not in globals():
    OPENROUTER_API_KEY = getpass("Введите ваш OpenRouter API key: ").strip()
    print("Ключ сохранён.")

def generate_answer(query, context_chunks):
    if not context_chunks:
        return "Не найдено релевантных фрагментов."
    context_text = "\n\n".join([c["text"] for c in context_chunks])
    prompt = f"""Ты — ассистент, отвечающий строго на основе предоставленного контекста.
Контекст состоит из фрагментов документов. Твои шаги:
1. Прочитай внимательно каждый фрагмент.
2. Найди конкретные факты, связанные с вопросом.
3. Составь краткий ответ, используя только эти факты (можно цитировать).
4. Если в контексте недостаточно информации для ответа, напиши ровно: 'Информации в предоставленных документах недостаточно.'
Не добавляй ничего от себя.

**Контекст:**
{context_text}

**Вопрос:** {query}

**Ответ:"""
    try:
        with OpenRouter(api_key=OPENROUTER_API_KEY) as client:
            response = client.chat.send(
                model=OPENROUTER_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=512,
                temperature=0.05,
            )
            return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Ошибка при вызове LLM: {str(e)}"

Введите ваш OpenRouter API key: ··········


### Пример полного цикла

Смоделируем пример полного цикла от начального гибридного поиска и переранжирования до получения ответа от модели. Гибридным поиском будем выбирать топ-15 кандидатов, а переранжированием оставим 5, на основе которых модель даст ответ.

In [ ]:
import time

query = "В Политике ООО «Озон Инвест» в отношении обработки персональных данных указано: в течение какого срока Оператор обязан уничтожить персональные данные после получения отзыва согласия от субъекта, если нет иных законных оснований для продолжения обработки?"
candidates = hybrid_search_rrf(query, top_k=15, k_rrf=60)
best_chunks = rerank(query, candidates, top_n=5)
start = time.time()
answer = generate_answer(query, best_chunks)
print(f"Время: {time.time()-start:.2f} сек")
print(f"Вопрос: {query}")
print(f"Ответ: {answer}\n")
print("Источники:")
for i, ch in enumerate(best_chunks, 1):
    print(f"{i}. {ch['text'][:200]}...")

Время: 11.91 сек
Вопрос: В Политике ООО «Озон Инвест» в отношении обработки персональных данных указано: в течение какого срока Оператор обязан уничтожить персональные данные после получения отзыва согласия от субъекта, если нет иных законных оснований для продолжения обработки?
Ответ: В Политике ООО «Озон Инвест» в отношении обработки персональных данных указано:  
«прекратить обработку персональных данных и уничтожить персональные данные в срок, не превышающий 30 (тридцати) дней с даты поступления указанного отзыва, если иное не предусмотрено соглашением между Оператором и субъектом персональных данных».  

**Ответ:** 30 (тридцати) дней.

Источники:
1. Юридические документы Ozon Условия обработки персональных данных В России Политика ООО «Озон Инвест» в отношении обработки персональных данных Политика ООО «Озон Инвест» в отношении обработки персонал...
2. дней с даты достижения цели обработки персональных данных, если иное не предусмотрено законодательством Российской Федерации; в сл

Модель ответила вполне корректно. Ответ дан, построен строго на основе контекста и полностью соответствует требованиям. Все источники соответствуют контексту. Модель дала ответ за 12 секунд, что в целом неплохо, но теоретически ее можно немного ускорить, поработав с параметрами запроса, такими как max_tokens, temperature.

# Тестирование модели

Теперь организуем пул тестов для того, чтобы получить метрики по работе модели. Было составлено 5 тестов на основе текстов.

In [ ]:
test_queries = [
    {
        "id": "q1",
        "query": "Согласно Master Supply Agreement между Veracier UK и Britannic Aerospace, каков максимальный размер совокупной ответственности (aggregate liability) поставщика? За какие нарушения эта ответственность не ограничена?",
        "expected_keywords": ["two (2) times the annual contract value", "IP infringement", "classified data breach", "unlimited"]
    },
    {
        "id": "q2",
        "query": "При первичном запуске программного обеспечения Personal Dose Tracker (MySQL) какие логин и пароль необходимо ввести для входа в качестве главного администратора?",
        "expected_keywords": ["admin", "admin"]
    },
    {
        "id": "q3",
        "query": "В Политике ООО «Озон Инвест» в отношении обработки персональных данных указано: в течение какого срока Оператор обязан уничтожить персональные данные после получения отзыва согласия от субъекта, если нет иных законных оснований для продолжения обработки?",
        "expected_keywords": ["30", "тридцати", "дней"]
    },
    {
        "id": "q4",
        "query": "В юридической заметке по патентному спору указано: каков срок исковой давности для предъявления иска согласно Limitation Act 1980 (Англия и Уэльс) и какая дата истечения этого срока в рассматриваемом деле?",
        "expected_keywords": ["six (6) years", "15 March 2028", "Limitation Act 1980"]
    },
    {
        "id": "q5",
        "query": "В ходе приемочного тестирования для Aperture Finance были согласованы целевые значения производительности. Какое минимальное целевое значение p99 латентности (в миллисекундах) и какое пороговое значение пропускной способности (throughput) в токенах в секунду были установлены как критерии «go/no-go»?",
        "expected_keywords": ["p99", "450", "ms", "12,000", "tokens/sec", "12k"]
    },
]

print(f"Загружено {len(test_queries)} тестовых запросов.")

Загружено 5 тестовых запросов.


При этом будем использовать стеммер (удаляет окончания), чтобы можно было учесть неточное попадание: например если другое окончание. Без стеммера метрики будут очень низкими: по субъективному прогнозу со стеммером 0.6-0.8, без него 0.1-0.3.

In [ ]:
import re
def simple_stem(word):
    word = word.lower()
    suffixes = ['ая', 'яя', 'ые', 'ие', 'ой', 'ей', 'ую', 'юю', 'ого', 'ему', 'ым', 'им', 'ом', 'ем',
                'ая', 'яя', 'ие', 'ые', 'ое', 'а', 'я', 'о', 'е', 'и', 'ы', 'у', 'ю', 'ь', 'й']
    for suffix in suffixes:
        if word.endswith(suffix):
            word = word[:-len(suffix)]
            break
    if len(word) < 3:
        return word
    return word
print("Стеммер готов.")

Стеммер готов.


Также сделаем функцию для нормализации текста.

In [ ]:
def normalize_text(text):
    words = re.findall(r'\b[а-яё]+\b', text.lower())
    stems = [simple_stem(w) for w in words]
    return set(stems)
print("Функция нормализации готова.")

Функция нормализации готова.


## Метрики

Напишем функцию для расчета метрик. Далее будем ее использовать для проверки. Здесь включены такие метрики, как Recall_fuzzy, time.

Функция recall_fuzzy. От recall@k она отличается тем, что при расчетах использует стеммер.

In [ ]:
def recall_fuzzy(query, expected_keywords, k=3):
    # Поиск и переранжирование
    candidates = hybrid_search_rrf(query, top_k=50, k_rrf=30)
    if 'rerank' in globals() and callable(rerank):
        best_chunks = rerank(query, candidates, top_n=k)
    else:
        best_chunks = [chunk for chunk, _ in candidates[:k]]

    if not best_chunks:
        return 0.0

    # Собираем все стемы из найденных чанков
    all_stems = set()
    for chunk in best_chunks:
        all_stems.update(normalize_text(chunk["text"]))

    # Проверяем каждую ожидаемую фразу
    found = 0
    for phrase in expected_keywords:
        phrase_stems = normalize_text(phrase)
        if phrase_stems.issubset(all_stems):
            found += 1

    recall = found / len(expected_keywords) if expected_keywords else 1.0
    return recall

print("Функция recall_fuzzy готова.")

Функция recall_fuzzy готова.


Напишем функцию для вызове всех метрик.

In [ ]:
def evaluate_rag(test_queries, top_k_retrieval=15, top_n_rerank=3, use_rerank=True):
    results = []
    for q in test_queries:
        start = time.time()

        # Гибридный поиск
        candidates = hybrid_search_rrf(q["query"], top_k=top_k_retrieval, k_rrf=60)

        # Переранжирование
        if use_rerank:
            best_chunks = rerank(q["query"], candidates, top_n=top_n_rerank)
        else:
            best_chunks = [chunk for chunk, _ in candidates[:top_n_rerank]]

        # recall_fuzzy со стеммером
        recall = recall_fuzzy(q["query"], q["expected_keywords"], k=3)

        # Генерация ответа
        answer = generate_answer(q["query"], best_chunks)

        elapsed = time.time() - start
        results.append({
            "id": q["id"],
            "recall": recall,
            "time_sec": elapsed,
            "answer": answer[:300]
        })
        print(f"{q['id']}: recall={recall:.2f}, time={elapsed:.2f}s")

    df = pd.DataFrame(results)
    print(f"Средний recall: {df['recall'].mean():.2f}")
    print(f"Среднее time: {df['time_sec'].mean():.2f} сек")
    return df

print("Функция evaluate_rag готова.")

Функция evaluate_rag готова.


В функции есть параметр use_rerank. Он задан на случай, если мы делаем проверку без переранжирования, например чтобы получить лучший замер времени ответа.

### Расчитаем метрики.

In [ ]:
df_results = evaluate_rag(test_queries, use_rerank=False)
df_results[['id', 'recall', 'time_sec']].head()

q1: recall=1.00, time=10.56s
q2: recall=1.00, time=10.13s
q3: recall=1.00, time=9.93s
q4: recall=1.00, time=10.42s
q5: recall=1.00, time=9.97s
Средний recall: 1.00
Среднее time: 10.20 сек


,id,recall,time_sec
0,q1,1.0,10.555677
1,q2,1.0,10.128630
2,q3,1.0,9.927233
3,q4,1.0,10.423922
4,q5,1.0,9.968073


Полученные метрики подтвердили правильную работу модели и целом алгоритма.

- recall = 1.00 говорит о том, что модель отвечает правильно в 100% случаев. примечательно, что это во много происходит из-за того, что мы добавили стеммер, без него значение было было 0.1-0.2 из-за неточного соответствия (ответ найден правильный, но неправильное окончание или что-то подобное).
- time_sec = 10.20 секунд - хороший показатель, который подтверждает, что модель "думает" около 10 секунд и возвращает корректный ответ.

Примечание: на ранних стадиях проекта модель думала 40-50 секунд. Проблема была решена оптимизацией функций работы с ней и корректировкой параметров, таких как размер чанка (экспериментировали с 128, 256, 384, 512, 768, 1024, 2048), размером перекрытия (50, 64, 128, 256, 384, 512, 1024), а также сравнивали гибрадный поиск с RRF и с alpha.